# Forest Fire Detection — Model Evaluation & Explainability
## Notebook 05: Test Set Evaluation, Confusion Matrix, Grad-CAM


In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_auc_score)

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
META_DIR = IMPL / "artifacts" / "metadata"
PLOTS    = IMPL / "artifacts" / "plots"
METRICS  = IMPL / "artifacts" / "metrics"
MODEL_DIR = IMPL / "models" / "image"
METRICS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)

with open(META_DIR / "data_splits.json") as f:
    splits = json.load(f)
with open(MODEL_DIR / "class_names.json") as f:
    cnames = json.load(f)
with open(MODEL_DIR / "model_metadata.json") as f:
    mmeta = json.load(f)

CLASS_NAMES  = cnames['class_names']
CLASS_TO_IDX = cnames['class_to_idx']
IDX_TO_CLASS = {int(k): v for k, v in cnames['idx_to_class'].items()}
SELECTED_MODEL = mmeta['model_name']
IMAGE_SIZE     = mmeta['image_size']
IMAGENET_MEAN  = mmeta['imagenet_mean']
IMAGENET_STD   = mmeta['imagenet_std']

print(f"Model: {SELECTED_MODEL} | Device: {DEVICE}")
print(f"Classes: {CLASS_NAMES}")


In [ ]:
# ── Rebuild model & load weights ──────────────────────────────────────
def build_model(model_name, num_classes=2):
    if model_name == 'EfficientNet-B0':
        m = models.efficientnet_b0(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'MobileNetV3':
        m = models.mobilenet_v3_small(weights=None)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    elif model_name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    elif model_name == 'ResNet18':
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

model = build_model(SELECTED_MODEL)
model.load_state_dict(torch.load(MODEL_DIR / "best_model.pth", map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print(f"Model loaded from: {MODEL_DIR}/best_model.pth")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
class FireDataset(torch.utils.data.Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        path, label = self.data[idx]
        try:
            img = Image.open(path).convert('RGB')
        except:
            img = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128,128,128))
        if self.transform: img = self.transform(img)
        return img, CLASS_TO_IDX[label], path

test_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
test_ds = FireDataset(splits['test'], test_transform)
test_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)

print(f"Test set: {len(test_ds)} images")


In [ ]:
# ── Evaluate on test set ──────────────────────────────────────────────
all_preds, all_labels, all_probs = [], [], []
inf_times = []

with torch.no_grad():
    for imgs, lbls, _ in test_loader:
        imgs = imgs.to(DEVICE)
        t0   = time.time()
        out  = model(imgs)
        inf_times.append((time.time() - t0) / len(imgs) * 1000)
        probs = F.softmax(out, dim=1)
        preds = out.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
fire_idx  = CLASS_TO_IDX['FIRE']
auc  = roc_auc_score(all_labels, all_probs[:, fire_idx])
inf_ms = np.mean(inf_times)

# Fire class specific recall (false negative rate)
cm = confusion_matrix(all_labels, all_preds)
fire_recall = cm[fire_idx, fire_idx] / cm[fire_idx].sum() if cm[fire_idx].sum() > 0 else 0.0

print("="*60)
print("TEST SET EVALUATION RESULTS")
print("="*60)
print(f"  Accuracy:          {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision:         {prec:.4f}")
print(f"  Recall (weighted): {rec:.4f}")
print(f"  F1-Score:          {f1:.4f}")
print(f"  ROC-AUC:           {auc:.4f}")
print(f"  FIRE class recall: {fire_recall:.4f}  (miss rate: {1-fire_recall:.4f})")
print(f"  Avg inference:     {inf_ms:.2f} ms/image")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

test_metrics = {
    'accuracy': round(float(acc), 4),
    'precision': round(float(prec), 4),
    'recall': round(float(rec), 4),
    'f1': round(float(f1), 4),
    'roc_auc': round(float(auc), 4),
    'fire_recall': round(float(fire_recall), 4),
    'inference_ms': round(float(inf_ms), 2),
    'test_samples': len(test_ds)
}
with open(METRICS / "image_test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
print(f"\nMetrics saved.")


In [ ]:
# Confusion matrix plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(cm, interpolation='nearest', cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {SELECTED_MODEL}', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_xticks(range(len(CLASS_NAMES))); axes[0].set_yticks(range(len(CLASS_NAMES)))
axes[0].set_xticklabels(CLASS_NAMES); axes[0].set_yticklabels(CLASS_NAMES)
plt.colorbar(im, ax=axes[0])
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=12, fontweight='bold')

# Bar chart of metrics
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
metric_vals  = [acc, prec, rec, f1, auc]
colors = ['#1E90FF','#32CD32','#FF8C00','#DC143C','#9370DB']
bars = axes[1].bar(metric_names, metric_vals, color=colors, edgecolor='black', alpha=0.85)
axes[1].set_ylim(0, 1.1)
axes[1].set_title('Performance Metrics — Test Set', fontweight='bold')
for bar, val in zip(bars, metric_vals):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS / "test_evaluation.png", dpi=100, bbox_inches='tight')
plt.close()
print("Evaluation plots saved.")


## 5. Grad-CAM Explainability

In [ ]:
class GradCAM:
    """Lightweight Grad-CAM implementation for CNN feature visualization."""
    def __init__(self, model, target_layer_name):
        self.model = model
        self.gradients = None
        self.activations = None
        self._register_hooks(target_layer_name)

    def _register_hooks(self, layer_name):
        target = dict(self.model.named_modules()).get(layer_name)
        if target is None:
            # Try to find the last conv layer automatically
            for name, module in self.model.named_modules():
                if isinstance(module, nn.Conv2d):
                    last_conv = (name, module)
            name, target = last_conv
            print(f"  Using last conv layer: {name}")
        target.register_forward_hook(self._save_activations)
        target.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, input, output):
        self.activations = output.detach()

    def _save_gradients(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        out = self.model(input_tensor)
        if class_idx is None:
            class_idx = out.argmax(dim=1).item()
        out[0, class_idx].backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam.squeeze().cpu().numpy()
        if cam.ndim == 0:
            cam = np.array([[cam]])
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

# Find target layer for selected model
def get_target_layer(model_name):
    return {
        'EfficientNet-B0': 'features.8.0',
        'MobileNetV3':     'features.12',
        'MobileNetV2':     'features.18.0',
        'ResNet18':        'layer4.1.conv2'
    }.get(model_name, None)

target_layer = get_target_layer(SELECTED_MODEL)
print(f"Grad-CAM target layer: {target_layer}")

# Build fresh model with grad support
model_gc = build_model(SELECTED_MODEL)
model_gc.load_state_dict(torch.load(MODEL_DIR / "best_model.pth", map_location='cpu'))
model_gc.eval()
gradcam = GradCAM(model_gc, target_layer)


In [ ]:
import random
random.seed(42)

# Denormalize helper
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

# Pick sample test images
test_paths = splits['test']
fire_samples   = [(p, l) for p, l in test_paths if l == 'FIRE'][:4]
nofire_samples = [(p, l) for p, l in test_paths if l == 'NO_FIRE'][:4]
samples = fire_samples[:2] + nofire_samples[:2]

fig, axes = plt.subplots(len(samples), 3, figsize=(12, 3.5*len(samples)))
fig.suptitle(f"Grad-CAM Explainability — {SELECTED_MODEL}", fontsize=13, fontweight='bold')

for row_idx, (img_path, true_label) in enumerate(samples):
    try:
        img_pil = Image.open(img_path).convert('RGB')
    except:
        continue

    inp = test_transform(img_pil).unsqueeze(0)

    # Grad-CAM
    cam, pred_idx = gradcam.generate(inp)
    pred_label = IDX_TO_CLASS[pred_idx]

    # Overlay
    img_resized = np.array(img_pil.resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
    heatmap = cm.jet(cam)[:, :, :3]
    overlay = 0.55 * img_resized + 0.45 * heatmap

    # Confidence
    with torch.no_grad():
        logits = model_gc(inp)
        probs  = F.softmax(logits, dim=1)[0]
    conf = probs[pred_idx].item()
    correct = "✓" if pred_label == true_label else "✗"

    axes[row_idx][0].imshow(img_resized)
    axes[row_idx][0].set_title(f"Original
True: {true_label}", fontsize=8)
    axes[row_idx][0].axis('off')

    axes[row_idx][1].imshow(cam, cmap='jet')
    axes[row_idx][1].set_title(f"Grad-CAM Heatmap
Hot=High attention", fontsize=8)
    axes[row_idx][1].axis('off')

    axes[row_idx][2].imshow(np.clip(overlay, 0, 1))
    c = 'green' if pred_label == true_label else 'red'
    axes[row_idx][2].set_title(f"Overlay {correct}
Pred: {pred_label} ({conf:.1%})", fontsize=8, color=c)
    axes[row_idx][2].axis('off')

plt.tight_layout()
plt.savefig(PLOTS / "gradcam_examples.png", dpi=100, bbox_inches='tight')
plt.close()
print("Grad-CAM visualization saved.")
print(f"\nNotebook 05 complete.")
print(f"Test Accuracy: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {auc:.4f}")
